# 🧠 Brain Tumor Segmentation Training

This notebook demonstrates how to train segmentation models using the modular codebase.

**Available Models:**
- UNet (with ResNet50 backbone)
- AttentionUNet
- ResUNetPP
- SwinUNet

**Available Loss Functions:**
- `bce_tversky` (default - BCE + Tversky combined)
- `dice`
- `dice_bce`
- `tversky`
- `focal_tversky`

## 1. Setup

In [ ]:
import os
import sys

# Add project root to path
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Suppress TensorFlow warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

## 2. Import Segmentation Module

In [ ]:
from src.segmentation import (
    SegmentationTrainer,
    build_segmentation_model,
    AVAILABLE_SEGMENTATION_MODELS,
    LOSS_FUNCTIONS,
)

print(f"\n✅ Available Models: {AVAILABLE_SEGMENTATION_MODELS}")
print(f"✅ Available Losses: {list(LOSS_FUNCTIONS.keys())}")

## 3. Configuration

Set your training parameters here:

In [ ]:
# ═══════════════════════════════════════════════════════════════
# TRAINING CONFIGURATION
# ═══════════════════════════════════════════════════════════════

# Model settings
MODEL_NAME = "UNet"           # Options: UNet, AttentionUNet, ResUNetPP, SwinUNet
LOSS_NAME = "bce_tversky"     # Options: dice, dice_bce, tversky, focal_tversky, bce_tversky
BACKBONE = "ResNet50"         # Backbone for encoder

# Training settings
EPOCHS = 200
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
IMG_SIZE = (256, 256)

# Data paths
TRAIN_IMAGES_DIR = "../data/brisc2025/segmentation_task/train/images"
TRAIN_MASKS_DIR = "../data/brisc2025/segmentation_task/train/masks"
TEST_IMAGES_DIR = "../data/brisc2025/segmentation_task/test/images"
TEST_MASKS_DIR = None  # Set if test masks are available

# Validation split
VAL_SPLIT = 0.2
RANDOM_STATE = 42

# Output directories
WEIGHTS_DIR = "../weights/segmentation"
LOGS_DIR = "../logs/segmentation"

print("✅ Configuration set!")

## 4. Initialize Trainer

In [ ]:
trainer = SegmentationTrainer(
    model_name=MODEL_NAME,
    loss_name=LOSS_NAME,
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    backbone=BACKBONE,
)

print(f"\n✅ Trainer initialized: {MODEL_NAME} with {LOSS_NAME} loss")

## 5. Prepare Data

In [ ]:
trainer.prepare_data(
    train_images_dir=TRAIN_IMAGES_DIR,
    train_masks_dir=TRAIN_MASKS_DIR,
    test_images_dir=TEST_IMAGES_DIR,
    test_masks_dir=TEST_MASKS_DIR,
    val_split=VAL_SPLIT,
    random_state=RANDOM_STATE,
    use_augmentation=True,
)

## 6. Build and Compile Model

In [ ]:
# Build model
model = trainer.build()

# Compile with loss and metrics
trainer.compile()

# Show model summary
model.summary()

## 7. Train Model

In [ ]:
history = trainer.train(
    epochs=EPOCHS,
    weights_dir=WEIGHTS_DIR,
    logs_dir=LOGS_DIR,
)

## 8. Visualize Training History

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Loss
axes[0, 0].plot(history.history['loss'], label='Train')
axes[0, 0].plot(history.history['val_loss'], label='Validation')
axes[0, 0].set_title('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Dice Coefficient
axes[0, 1].plot(history.history['dice_coefficient'], label='Train')
axes[0, 1].plot(history.history['val_dice_coefficient'], label='Validation')
axes[0, 1].set_title('Dice Coefficient')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# IoU
axes[1, 0].plot(history.history['iou_score'], label='Train')
axes[1, 0].plot(history.history['val_iou_score'], label='Validation')
axes[1, 0].set_title('IoU Score')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Sensitivity
axes[1, 1].plot(history.history['sensitivity'], label='Train')
axes[1, 1].plot(history.history['val_sensitivity'], label='Validation')
axes[1, 1].set_title('Sensitivity')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle(f'{MODEL_NAME} Training Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{LOGS_DIR}/{MODEL_NAME}_{LOSS_NAME}_training_curves.png', dpi=150)
plt.show()

## 9. Sample Predictions

In [ ]:
import numpy as np

# Get a batch from validation set
for images, masks in trainer.val_ds.take(1):
    # Predict
    predictions = model.predict(images[:4])
    
    # Visualize
    fig, axes = plt.subplots(4, 4, figsize=(16, 16))
    
    for i in range(4):
        # Original image
        axes[i, 0].imshow(images[i].numpy())
        axes[i, 0].set_title('Original')
        axes[i, 0].axis('off')
        
        # Ground truth mask
        axes[i, 1].imshow(masks[i].numpy()[:,:,0], cmap='gray')
        axes[i, 1].set_title('Ground Truth')
        axes[i, 1].axis('off')
        
        # Predicted mask
        axes[i, 2].imshow(predictions[i][:,:,0], cmap='gray')
        axes[i, 2].set_title('Prediction')
        axes[i, 2].axis('off')
        
        # Overlay
        overlay = images[i].numpy().copy()
        pred_mask = (predictions[i][:,:,0] > 0.5).astype(np.float32)
        overlay[:,:,0] = np.clip(overlay[:,:,0] + pred_mask * 0.5, 0, 1)
        axes[i, 3].imshow(overlay)
        axes[i, 3].set_title('Overlay')
        axes[i, 3].axis('off')
    
    plt.suptitle('Sample Predictions', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{LOGS_DIR}/{MODEL_NAME}_{LOSS_NAME}_predictions.png', dpi=150)
    plt.show()

## 10. Save Model

In [ ]:
trainer.save()
print("\n✅ Training complete! Model saved.")

---

## Alternative: Train via Command Line

You can also train using the CLI script:

```bash
# Default training (UNet + BCE-Tversky)
python scripts/train_segmentor.py

# Train Attention U-Net
python scripts/train_segmentor.py --model AttentionUNet

# Train with specific loss
python scripts/train_segmentor.py --model UNet --loss focal_tversky

# Train with config file
python scripts/train_segmentor.py --config configs/segmentation_config.yaml
```